**Initial Set-Up**

1. Install Anaconda3 using GUI installer/install Miniforge

2. Run installation commands to install Mamba

    conda config --add channels conda-forge

    conda update -n base --all

    conda install -n base mamba


3. Use Mamba to install PyImageJ
    mamba install -c conda-forge pyimagej openjdk=11

**OR, use Pip**

    Install Python 3
    
    Install OpenJDK 8 or OpenJDK 11

    Install Maven

    Run: pip install pyimagej


**Errors Encountered Previously and Currently**

Install Java
   IF: You recieve an error indicating the program's inability to resolve Java, 
   then install the official Java Development Kit (JDK from the website) 
   https://www.oracle.com/java/technologies/downloads/?er=221886#jdk25. 

Original ImageJ not available    
 - Do not change initialization line. 

Re-running cell with initialization code (see below) leads to problems with initialization.
 - Guarding implemented ("if 'ij' not in globals()...")

 **Todo** 
  - Take measurements from an image and append to a .csv file
  - Implement looping through a folder. 
  - Dump csv --> .pkl file

In [ ]:
# ! mamba remove -n base pyimagej
# ! pip install pyimagej

from IPython.display import Image, display
from pathlib import Path
import jpype
import imagej
import scyjava
from scyjava import jimport
import pandas as pd
import numpy as np
import os

print("Dependencies present")

Dependencies present


In [2]:
# Set Random Image for Testing
test_image_pth = "content/Images1/noise 3 21001.0.png"

# Pre-sets
path_in1 = "./content/Images1/"
path_in2 = "./content/Images2/"
path_in3 = "./content/Images3/"

path_out_csv = "./data/feats_extracted.csv"

# Set Java heap size = 6 gb.
# scyjava.config.add_option('-Xmx6g')

# Folder containing PNG images
# Select one of the input paths (path_in1, path_in2, path_in3)
input_dir = path_in1
output_csv = Path(path_out_csv)

print("I/O Set.")

I/O Set.


In [ ]:
print(f"Analyzing measurements on all images from folder {input_dir}")


# Initialize ImageJ
# Java Virtual Machine Guarding
if 'ij' not in globals():
    ij = imagej.init('sc.fiji:fiji', headless=False, add_legacy=True)
else:
    print("JVM instance already running.")

print(f"Ensure ImageJ Legacy layer is available. Status: {ij.legacy.isActive()}")

counter = 0
for file in os.listdir(input_dir):
    counter += 1
    
    if counter % 1000 == 0:
        print(counter)

    # Can be deleted for production purposes later. Implemented for efficiency.
    if counter > 2000:
        break

    full_path = str(input_dir) + '/' + file

    # Load test image
    dataset = ij.IJ.openImage(full_path)

    # Display the image within Jupyter Lab for intuitive purposes 
    #display(Image(full_path))

    # Select entire image
    ij.IJ.run(dataset, "Select All", "")

    # Set Measurements
    ij.IJ.run("Set Measurements...", """area mean standard modal min centroid center perimeter bounding fit shape feret's integrated median skewness kurtosis area_fraction stack display add redirect=None decimal=3""")

    # Extract Measurements from the image
    ij.IJ.run(dataset, "Measure", "")   

print("Measurments complete.") 
print("Do not close ImageJ Results window before compiling the rest of the program.")
print("Current limitation: do not run this block with all three folders because dynamic folder column addition is not yet implemented. Run it with one folder, save to dataframe, then rerun.")

The headless flag of imagej.init is deprecated. Use the mode argument instead.


Analyzing measurements on all images from folder ./content/Images1/


OpenJDK 64-Bit Server VM warning: Attempt to protect stack guard pages failed.
OpenJDK 64-Bit Server VM warning: Attempt to deallocate stack guard pages failed.


RuntimeError: Can't find org.jpype.jar support library

In [18]:
# Get Results 
ResultsTable = ij.ResultsTable.getResultsTable()

# Returns tab delimited. .split ensure grouping
cols = list(ResultsTable.getColumnHeadings().split('\t'))
cols.remove(' ')

all_data = {}

for col in cols:
    col = str(col)
    if col == 'Label':
        col_data = [ResultsTable.getStringValue(col, i) for i in range(ResultsTable.size())]
        for i in range(len(col_data)):
            col_data[i] = str(col_data[i])
        all_data[col] = col_data
    else:
        col_data = ResultsTable.getColumn(col)
        all_data[col] = col_data

feats_extracted_df = pd.DataFrame(data=all_data)
feats_extracted_df.rename(columns={'Label' : 'Path'}, inplace=True)
feats_extracted_df['Folder'] = f'{input_dir}'

feats_extracted_df

,Path,Area,Mean,StdDev,Mode,Min,Max,X,Y,XM,...,RawIntDen,Slice,FeretX,FeretY,FeretAngle,MinFeret,AR,Round,Solidity,Folder
0,noise 3 21001.0.png,40000.0,2.556375,25.403872,0.0,0.0,255.0,100.0,100.0,98.896509,...,102255.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
1,noise 3 21002.0.png,40000.0,0.000000,0.000000,0.0,0.0,0.0,100.0,100.0,100.000000,...,0.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
2,noise 3 21003.0.png,40000.0,255.000000,0.000000,255.0,255.0,255.0,100.0,100.0,100.000000,...,10200000.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
3,noise 3 21004.0.png,40000.0,0.000000,0.000000,0.0,0.0,0.0,100.0,100.0,100.000000,...,0.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
4,noise 3 21005.0.png,40000.0,9.925875,49.321759,0.0,0.0,255.0,100.0,100.0,100.628452,...,397035.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,noise 3 22996.0.png,40000.0,255.000000,0.000000,255.0,255.0,255.0,100.0,100.0,100.000000,...,10200000.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
1996,noise 3 22997.0.png,40000.0,255.000000,0.000000,255.0,255.0,255.0,100.0,100.0,100.000000,...,10200000.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
1997,noise 3 22998.0.png,40000.0,255.000000,0.000000,255.0,255.0,255.0,100.0,100.0,100.000000,...,10200000.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/
1998,noise 3 22999.0.png,40000.0,36.739125,89.548389,0.0,0.0,255.0,100.0,100.0,98.937619,...,1469565.0,1.0,0.0,0.0,135.0,200.0,1.0,1.0,1.0,./content/Images1/


In [ ]:
## GET PARTICLE DATA
## Current issues/status: I am able to get particle data (num_spots, feret, area, etc)
## However, I am not sure how to take the summary dialogue and rip the data from it.
## Next steps: rip data from summary dialogue; loop through all images and add images to particle dataframe
## Final steps: merge particles df and feats_extracted_df by path
## and extract to CSV

# Open Image Dataset
dataset = ij.IJ.openImage(test_image_pth)

# Convert to ImagePlus for Compat.
dataset = ij.py.to_imageplus(dataset)

# Select entire image
ij.IJ.run(dataset, "Select All", "")

# Analyze Particles
ij.IJ.run("""Analyze Particles...""", "display clear summarize overlay record")

ResultsTable = ij.ResultsTable.getResultsTable()

# Returns tab delimited. .split ensure grouping
cols = list(ResultsTable.getColumnHeadings().split('\t'))
cols.remove(' ')

all_data = {}

for col in cols:
    col = str(col)
    if col == 'Slice':
        col_data = [ResultsTable.getStringValue(col, i) for i in range(ResultsTable.size())]
        for i in range(len(col_data)):
            col_data[i] = str(col_data[i])
        all_data[col] = col_data
    else:
        col_data = ResultsTable.getColumn(col)
        all_data[col] = col_data

particle_summary_df = pd.DataFrame(data=all_data)
particle_summary_df.rename(columns={'Slice' : 'Path'}, inplace=True)

particle_summary_df


,Path,Area,Mean,StdDev,Mode,Min,Max,X,Y,XM,...,Slice,FeretX,FeretY,FeretAngle,MinFeret,AR,Round,Solidity,XStart,YStart
0,noise 3 21001.0.png,7.0,255.0,0.0,255.0,255.0,255.0,12.785714,0.928571,12.785714,...,1.0,11.0,2.0,26.565051,2.000000,1.935706,0.516607,0.933333,11.0,0.0
1,noise 3 21001.0.png,5.0,255.0,0.0,255.0,255.0,255.0,67.700000,0.900000,67.700000,...,1.0,66.0,0.0,146.309932,2.000000,1.552986,0.643921,0.909091,66.0,0.0
2,noise 3 21001.0.png,1.0,255.0,0.0,255.0,255.0,255.0,95.500000,0.500000,95.500000,...,1.0,95.0,0.0,135.000000,1.000000,1.000000,1.000000,1.000000,95.0,0.0
3,noise 3 21001.0.png,6.0,255.0,0.0,255.0,255.0,255.0,158.500000,1.000000,158.500000,...,1.0,157.0,0.0,146.309932,2.000000,1.500000,0.666667,1.000000,157.0,0.0
4,noise 3 21001.0.png,10.0,255.0,0.0,255.0,255.0,255.0,131.800000,12.800000,131.800000,...,1.0,130.0,14.0,26.565051,3.535534,1.241080,0.805750,0.800000,131.0,11.0
5,noise 3 21001.0.png,9.0,255.0,0.0,255.0,255.0,255.0,184.500000,15.500000,184.500000,...,1.0,183.0,14.0,135.000000,3.000000,1.000000,1.000000,1.000000,183.0,14.0
6,noise 3 21001.0.png,11.0,255.0,0.0,255.0,255.0,255.0,43.318182,18.318182,43.318182,...,1.0,43.0,16.0,116.565051,4.000000,1.035071,0.966118,0.846154,43.0,16.0
7,noise 3 21001.0.png,10.0,255.0,0.0,255.0,255.0,255.0,78.500000,25.700000,78.500000,...,1.0,77.0,24.0,116.565051,3.000000,1.235649,0.809291,0.909091,77.0,24.0
8,noise 3 21001.0.png,9.0,255.0,0.0,255.0,255.0,255.0,110.500000,25.500000,110.500000,...,1.0,109.0,24.0,135.000000,3.000000,1.000000,1.000000,1.000000,109.0,24.0
9,noise 3 21001.0.png,10.0,255.0,0.0,255.0,255.0,255.0,171.700000,39.500000,171.700000,...,1.0,170.0,38.0,153.434949,3.000000,1.235649,0.809291,0.909091,170.0,38.0


In [5]:
feats_extracted_df.to_csv(path_out_csv)